In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
from datetime import datetime

# Base de datos vacía
horarios_df = pd.DataFrame(columns=["Día", "Hora Inicio", "Hora Fin", "Docente", "Clase", "Aula"])

# Opciones posibles
dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes']
horas = ['08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00', '17:00']
aulas = ['A101', 'B202', 'C303']

# Widgets de entrada
dia = widgets.Dropdown(options=dias, description="Día")
hora_inicio = widgets.Dropdown(options=horas, description="Inicio")
hora_fin = widgets.Dropdown(options=horas, description="Fin")
docente = widgets.Text(description="Docente")
clase = widgets.Text(description="Clase")
aula = widgets.Dropdown(options=aulas, description="Aula")

# Botones
btn_agregar = widgets.Button(description="Agregar clase")
btn_ver = widgets.Button(description="Ver horario")
salida = widgets.Output()

# Función para verificar traslape y conflictos
def hay_conflicto(fila_nueva, horarios):
    nuevo_inicio = datetime.strptime(fila_nueva["Hora Inicio"], "%H:%M")
    nuevo_fin = datetime.strptime(fila_nueva["Hora Fin"], "%H:%M")

    for _, fila in horarios.iterrows():
        if fila["Día"] != fila_nueva["Día"]:
            continue  # solo comparar clases del mismo día

        inicio = datetime.strptime(fila["Hora Inicio"], "%H:%M")
        fin = datetime.strptime(fila["Hora Fin"], "%H:%M")

        traslape = (
            (inicio <= nuevo_inicio < fin) or
            (inicio < nuevo_fin <= fin) or
            (nuevo_inicio <= inicio and nuevo_fin >= fin)
        )

        if traslape:
            if fila_nueva["Docente"] == fila["Docente"]:
                return f"❌ Conflicto: el docente '{fila['Docente']}' ya tiene clase en ese horario."

            if fila_nueva["Aula"] == fila["Aula"]:
                return f"❌ Conflicto: el aula '{fila['Aula']}' ya está ocupada en ese horario."

            if fila_nueva["Clase"] == fila["Clase"] and fila_nueva["Docente"] != fila["Docente"]:
                return f"❌ Conflicto: la clase '{fila['Clase']}' ya está asignada con otro docente en ese horario."

    return None  # No hay conflictos

# Función principal para agregar clases
def agregar_clase(_):
    salida.clear_output()

    # Validar campos vacíos
    if not docente.value.strip() or not clase.value.strip():
        with salida:
            print("⚠️ Por favor completa todos los campos.")
        return

    # Validar formato de hora
    try:
        hi = datetime.strptime(hora_inicio.value, "%H:%M")
        hf = datetime.strptime(hora_fin.value, "%H:%M")
    except:
        with salida:
            print("⚠️ Error en formato de hora.")
        return

    # Validar que hora de inicio < hora de fin
    if hi >= hf:
        with salida:
            print("⚠️ La hora de fin debe ser mayor que la hora de inicio.")
        return

    # Crear fila nueva
    fila_nueva = {
        "Día": dia.value,
        "Hora Inicio": hora_inicio.value,
        "Hora Fin": hora_fin.value,
        "Docente": docente.value.strip(),
        "Clase": clase.value.strip(),
        "Aula": aula.value
    }

    # Verificar conflictos
    conflicto = hay_conflicto(fila_nueva, horarios_df)

    with salida:
        if conflicto:
            print(conflicto)
        else:
            horarios_df.loc[len(horarios_df)] = fila_nueva
            print("✅ Clase agregada correctamente.")

# Función para mostrar horario ordenado por día y hora
def ver_horario(_):
    salida.clear_output()
    with salida:
        if horarios_df.empty:
            print("⚠️ No hay clases asignadas.")
        else:
            orden_dias = {"Lunes": 0, "Martes": 1, "Miércoles": 2, "Jueves": 3, "Viernes": 4}
            df = horarios_df.copy()
            df["Orden Día"] = df["Día"].map(orden_dias)
            df = df.sort_values(by=["Orden Día", "Hora Inicio"])
            display(df.drop(columns="Orden Día"))

# Conectar botones a funciones
btn_agregar.on_click(agregar_clase)
btn_ver.on_click(ver_horario)

# Mostrar interfaz
display(dia, hora_inicio, hora_fin, aula, docente, clase, btn_agregar, btn_ver, salida)


Dropdown(description='Día', options=('Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes'), value='Lunes')

Dropdown(description='Inicio', options=('08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00'…

Dropdown(description='Fin', options=('08:00', '09:00', '10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '…

Dropdown(description='Aula', options=('A101', 'B202', 'C303'), value='A101')

Text(value='', description='Docente')

Text(value='', description='Clase')

Button(description='Agregar clase', style=ButtonStyle())

Button(description='Ver horario', style=ButtonStyle())

Output()